# Vježbe 8 — Neuralne mreže: MLP (Multi-Layer Perceptron)
**Predmet:** Rudarenje podataka - I. godina - II ciklus - Softversko inženjerstvo  
**Asistent:** ass. mr. Narcisa Hadžajlić  
**Datum:** 13.4.2026.

---

## Šta smo do sada naučili 

- Šta je neuron / node: osnovna jedinica neuronske mreže i može se zamisliti kao mali kalkulator koji prima ulazne podatke, obrađuje ih i šalje rezultat dalje.
- Šta je layer (sloj): skup neurona koji rade paralelno na istom nivou obrade podataka. (Tipično 3 vrste slojeva: **Input Layer (Ulazni sloj)**, **Hidden Layers (Skriveni slojevi)**, **Output Layer (Izlazni sloj)**. 
- Šta je epoha (epoch): jedan potpuni prolaz cijelog skupa podataka za trening kroz neuronsku mrežu (i naprijed i nazad). Npr. za 1.000 slika pasa, jedna epoha je gotova kada mreža "vidi" i pokuša naučiti nešto iz svih tih 1.000 slika. Učenje obično zahtijeva više epoha (desetine, stotine ili hiljade) jer mreža u prvom prolazu pravi mnogo grešaka, pa kroz ponavljanje postepeno optimizuje svoje parametre.
- Šta je batch processing: obrada u serijama, tehnika gdje ne šaljemo sve podatke odjednom u mrežu, niti ih šaljemo jedan po jedan, već u manjim grupama.

## Šta radimo danas

1. Kako podaci **putuju kroz mrežu** — forward pass
2. Kako mreža **uči iz grešaka** — backpropagation (intuicija, bez maths!)
3. Implementacija MLP u `sklearn` i `Keras`
4. Usporedba MLP s algoritmima koje smo već znali (SVM, Decision Tree)
5. **Zadatak na času** (3 boda)


---
## 1. Kako podaci putuju kroz mrežu — Forward Pass

Zamislite fabriku:
- **Input layer** = sirovine ulaze
- **Hidden layers** = svaki radnik nešto uradi sa sirovinom
- **Output layer** = gotov proizvod izlazi

Svaka veza između neurona ima **težinu (weight)**. Neuron prima inputs, množi ih s težinama, sabere, propusti kroz **aktivacijsku funkciju** i šalje dalje.

```
output = aktivacija( w1*x1 + w2*x2 + w3*x3 + bias )
```

Na kraju mreže dobijemo predikciju. Ako je predikcija pogrešna — tu nastupa **backpropagation**.


---
## 2. Kako mreža uči — Backpropagation (intuicija)

**Analogija:** Učite voziti i pogriješite na okuci.
- Instruktor vam kaže koliko ste pogriješili → to je **loss (greška)**
- Vi prilagodite ruke na volanu → to su **težine (weights)**
- Koliko agresivno prilagodite → to je **learning rate**

Backpropagation ide **unazad** kroz mrežu i govori svakom neuronu: *"Ti si toliko odgovoran za ovu grešku — prilagodi se toliko!"*

**Gradijentni spust (gradient descent):**  
Zamislite brdo u magli. Stojite na vrhu i hoćete doći do dna (minimalna greška).  
Svaki korak idete u smjeru pada → to je jedna iteracija učenja.

> **Ključna poruka:** Mreža ne zna ništa unaprijed. Ona **iterativno ispravlja sebe** kroz stotine/hiljade prolaza kroz podatke (epohe).


---
## 3. Implementacija 


In [1]:
# Importi — sve što nam treba danas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')

print("Sve je uvezeno!")

Sve je uvezeno!


### 3.1 Dataset — Iris (nastavak)

Iris.

In [2]:
# Učitavanje Iris dataseta
df = pd.read_csv("iris.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'iris.csv'

In [ ]:
df.info()

In [ ]:
df['species'].value_counts()

### 3.2 Priprema podataka

**Važno za neuralne mreže:** Neuralne mreže su OSJETLJIVE na skalu podataka!  
Feature s vrijednostima 0-1 i feature s vrijednostima 0-10000 neće biti ravnopravni.  
Zato koristimo **StandardScaler** — svodi sve feateure na istu skalu (mean=0, std=1).

In [ ]:
X = df.drop('species', axis=1)
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Skaliranje — OBAVEZNO za neuralne mreže!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform na trainu
X_test_scaled  = scaler.transform(X_test)         # SAMO transform na testu (ne smijemo fitati na testu!)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"\nPrimjer — originalni vs. skalirani (prve 3 vrste):")
print("Original:", X_train.values[0].round(2))
print("Scaled:  ", X_train_scaled[0].round(2))

### 3.3 MLP s sklearn — MLPClassifier

Počnimo s najjednostavnijom mrežom i razumijemo šta svaki parametar znači.

In [ ]:
# Najjednostavniji MLP
# hidden_layer_sizes=(100,)  → jedan hidden layer s 100 neurona
# max_iter=500               → maksimalno 500 epoha učenja
# random_state=42            → reproducibilnost

mlp_simple = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=500,
    random_state=42
)

mlp_simple.fit(X_train_scaled, y_train)
y_pred_simple = mlp_simple.predict(X_test_scaled)

f1_simple = f1_score(y_test, y_pred_simple, average='weighted')
print(f"F1 score (jednostavni MLP): {f1_simple:.4f}")

In [ ]:
# Koliko layera i neurona ima naša mreža?
print(f"Broj layera (hidden): {len(mlp_simple.hidden_layer_sizes)}")
print(f"Broj neurona po layeru: {mlp_simple.hidden_layer_sizes}")
print(f"Broj epoha koje su bile potrebne: {mlp_simple.n_iter_}")
print(f"Konvergiralo: {mlp_simple.n_iter_ < 500}")

### 3.4 Compare and contrast — šta se desi kad mijenjamo arhitekturu mreže?

Šta se desi s različitim konfiguracijama?

In [ ]:
# Testiramo različite arhitekture
konfiguracije = {
    "1 layer, 10 neurona":   (10,),
    "1 layer, 100 neurona":  (100,),
    "2 layera (50, 25)":     (50, 25),
    "3 layera (100, 50, 25)":(100, 50, 25),
    "Plitko i široko (200,)": (200,),
}

rezultati = {}

for naziv, arhitektura in konfiguracije.items():
    model = MLPClassifier(
        hidden_layer_sizes=arhitektura,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    f1 = f1_score(y_test, y_pred, average='weighted')
    rezultati[naziv] = {
        'f1': round(f1, 4),
        'epohe': model.n_iter_
    }
    print(f"{naziv:35s} → F1: {f1:.4f}  (epohe: {model.n_iter_})")

In [ ]:
# Vizualizacija rezultata
nazivi = list(rezultati.keys())
f1_vrijednosti = [r['f1'] for r in rezultati.values()]

plt.figure(figsize=(10, 5))
bars = plt.barh(nazivi, f1_vrijednosti, color='steelblue', edgecolor='white')
plt.xlabel('F1 Score (weighted)')
plt.title('MLP arhitekture — usporedba F1 Scorea')
plt.xlim(0.8, 1.01)

for bar, val in zip(bars, f1_vrijednosti):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 3.5 Kriva učenja — kako mreža napreduje kroz epohe?

In [ ]:
# Pratimo loss kroz epohe
mlp_s_historijom = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=500,
    random_state=42,
    verbose=False  # promijenite na True da vidite ispis po epohi
)
mlp_s_historijom.fit(X_train_scaled, y_train)

plt.figure(figsize=(9, 4))
plt.plot(mlp_s_historijom.loss_curve_, color='coral', linewidth=2)
plt.xlabel('Epoha')
plt.ylabel('Loss (greška)')
plt.title('Kriva učenja — loss pada kako mreža uči')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPoček: loss = {mlp_s_historijom.loss_curve_[0]:.4f}")
print(f"Kraj:  loss = {mlp_s_historijom.loss_curve_[-1]:.4f}")

### 3.6 MLP vs. algoritmi koje smo već znali

Je li MLP bolji od Decision Tree i SVM na ovom datasetu?

In [ ]:
modeli = {
    'Decision Tree':  DecisionTreeClassifier(random_state=42),
    'SVM (RBF)':      SVC(kernel='rbf', random_state=42),
    'MLP (100, 50)':  MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42),
}

print(f"{'Model':20s}  F1 Score")
print("-" * 35)

for naziv, model in modeli.items():
    # Decision Tree ne treba skaliranje, ali neće mu smetati
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{naziv:20s}  {f1:.4f}")

### 3.7 Detaljan izvještaj za naš MLP

In [ ]:
# Classification report za best MLP
best_mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
best_mlp.fit(X_train_scaled, y_train)
y_pred_best = best_mlp.predict(X_test_scaled)

print(classification_report(y_test, y_pred_best))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)
klase = best_mlp.classes_

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(klase)))
ax.set_yticks(range(len(klase)))
ax.set_xticklabels(klase, rotation=45, ha='right')
ax.set_yticklabels(klase)

for i in range(len(klase)):
    for j in range(len(klase)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)

ax.set_xlabel('Predikcija')
ax.set_ylabel('Stvarna klasa')
ax.set_title('Confusion Matrix — MLP (100, 50)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---
## 4. Kratko o Keras / TensorFlow

sklearn MLPClassifier je dobar za učenje i brze eksperimente.  
Za ozbiljne projekte (i za CNN, LSTM, GAN koje ćemo raditi) koristimo **Keras**.

> Razlika je kao između auto-mjenjača i manualnog: sklearn radi puno toga automatski, Keras daje potpunu kontrolu.

In [ ]:
# Isti problem, isti dataset — ali sada u Keras
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.preprocessing import LabelEncoder
    import numpy as np

    # Keras treba numeričke labele
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc  = le.transform(y_test)

    # Definišemo mrežu
    model_keras = keras.Sequential([
        layers.Input(shape=(4,)),          # 4 input featuri (Iris ima 4 kolone)
        layers.Dense(100, activation='relu'),  # hidden layer 1: 100 neurona, ReLU aktivacija
        layers.Dense(50,  activation='relu'),  # hidden layer 2: 50 neurona
        layers.Dense(3,   activation='softmax') # output: 3 klase (softmax za multiclass)
    ])

    # Kompajliranje — biramo optimizer i loss funkciju
    model_keras.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model_keras.summary()

except ImportError:
    print("TensorFlow nije instaliran. Instalirajte s: pip install tensorflow")
    print("Za danas ćemo raditi s sklearn MLPClassifier.")

In [ ]:
# Treniranje Keras modela
try:
    historija = model_keras.fit(
        X_train_scaled, y_train_enc,
        epochs=100,           # 100 epoha
        batch_size=16,        # 16 uzoraka po batchu
        validation_split=0.2, # 20% traina ide kao validacioni set
        verbose=0             # tihi mod — promijenite na 1 za ispis po epohi
    )

    # Evaluacija
    y_pred_keras = np.argmax(model_keras.predict(X_test_scaled, verbose=0), axis=1)
    f1_keras = f1_score(y_test_enc, y_pred_keras, average='weighted')
    print(f"F1 score (Keras MLP): {f1_keras:.4f}")

    # Kriva učenja
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(historija.history['loss'], label='Train loss', color='coral')
    ax1.plot(historija.history['val_loss'], label='Val loss', color='steelblue')
    ax1.set_title('Loss kroz epohe')
    ax1.set_xlabel('Epoha')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(historija.history['accuracy'], label='Train acc', color='coral')
    ax2.plot(historija.history['val_accuracy'], label='Val acc', color='steelblue')
    ax2.set_title('Accuracy kroz epohe')
    ax2.set_xlabel('Epoha')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

except NameError:
    print("Keras model nije kreiran — vidjeti poruku iznad.")

---
## 5. Sažetak pojmova

| Koncept | Šta znači |
|---|---|
| **Forward pass** | Podaci putuju od inputa do outputa kroz layere |
| **Loss** | Mjera koliko je predikcija pogrešna |
| **Backpropagation** | Greška se propagira unazad, težine se ispravljaju |
| **Learning rate** | Koliko agresivno se ispravljamo po svakom koraku |
| **Epoha** | Jedan cijeli prolaz kroz sve trening podatke |
| **Batch** | Koliko uzoraka odjednom ulazi u mrežu |
| **StandardScaler** | Obavezan preprocessing za neuralne mreže |
| **Softmax** | Aktivacija za output u multiclass klasifikaciji |
| **ReLU** | Najpopularnija aktivacija u hidden layerima |

---
## ZADATAK - IZRADA NA ČASU — 3 BODA

**Rok:** Do kraja časa - commit na GitHub obavezan do 23:59 13.04.2026.)

### Treba uraditi sljedeće:

1. **Pronađite vlastiti dataset** na [kaggle.com](https://kaggle.com) koji ima:
   - Numeričke feature-e (barem 3-4 kolone)
   - Klasifikacijski target (kategoričke labele)
   - Preporučeni tematski domen: sport, muzika, health, gaming, food — nešto što vas zanima!

2. **Istrenirajte MLP** (sklearn ili Keras — vaš izbor) i pokažite:
   - Preprocessing s `StandardScaler`
   - Barem 2 različite arhitekture (uporedite ih!)
   - F1 score i classification report za najbolji model
   - Kriva učenja (loss_curve_ ili Keras historija)

3. **Napišite komentar** (markdown ćelija na kraju): Šta ste primijetili? Je li veća mreža uvijek bolja? Koliko epoha je trebalo?

### Urađeni zadtak postavite na:
```
GitHub repo → 2026/Zadace/Vj8/<VasePrezimeIme>/vjezba_8_zadatak.ipynb
```

### Bodovanje:
- 1 bod — Kod radi, MLP je istreniran, F1 score je prikazan
- 1 bod — Usporedba min. 2 arhitekture, kriva učenja
- 1 bod — Vaš dataset (ne Iris!), komentar s vlastitom analizom

**Bonus pitanje za razmišljanje:** Šta se desi s F1 scoreom ako zaboravite da primijenite StandardScaler? Probajte!